# Responsiveness: load, analyze, plot

Edit the settings below and run the notebook from top to bottom. The loader samples query cells within each tissue **before** combining files, retains every sampled context and its original baseline, and prefixes IDs with the tissue name. Helper functions live in `responsiveness_utils.py`. The previous exploratory notebook is saved as `Responsiveness_exploration.ipynb`.

In [3]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import seaborn as sns

HELPER_DIR = Path('/home/jwinchel/Syndri/context_analysis')
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))
from responsiveness_utils import (
    prediction_files, load_results, add_gene_annotations,
    analyze_results, plot_response_distribution, plot_slope_clustermap,
)

PREDICTION_DIR = Path('/work/magroup/skrieger/Syndri/data/datasets/breast_xenium/results/responsiveness/predictions')
CELL_TYPE = 'cytotoxic'       # 'cytotoxic', 'cd4_memory', or 'treg'
TISSUES = None                # e.g. ['GSM9509172', 'GSM9509173']; None means all completed files
MAX_FOCAL_CELLS_PER_TISSUE = 1  # None means all query cells
MAX_REPEATS = 1             # None means all repeats
GENES = []                  # e.g. ['CD3D', 'GZMB']; empty selects variable genes
N_AUTO_GENES = 12
LOGIT = True                # False keeps probabilities on the 0–1 scale
RANDOM_SEED = 42
ANNOTATION_FILE = Path('/work/magroup/skrieger/Syndri/data/datasets/breast_xenium/raw/single-cell/41592_2026_3089_MOESM4_ESM.xlsx')

sns.set_theme(style='whitegrid')

## Load selected tissues

`prediction_files` ignores prepared intermediates. `load_results` reads only the chosen query rows from each prediction file into memory. Missing neighborhood cell types are filled with zero before combining tissues.

In [4]:
files = prediction_files(PREDICTION_DIR, CELL_TYPE, TISSUES)
print(f'Found {len(files)} prediction files:')
for path in files:
    print(' ', path.name)

adata = load_results(
    files,
    max_focal_cells_per_tissue=MAX_FOCAL_CELLS_PER_TISSUE,
    max_repeats=MAX_REPEATS,
    seed=RANDOM_SEED,
)
if ANNOTATION_FILE.is_file():
    add_gene_annotations(adata, ANNOTATION_FILE)
print(f'Loaded {adata.n_obs:,} query-context rows, {adata.n_vars} shared genes')
display(adata.obs.groupby('tissue', observed=True).agg(
    rows=('focal_cell_id', 'size'), queries=('focal_cell_id', 'nunique'), repeats=('sampling_repeat', 'nunique')
))

Found 19 prediction files:
  GSM9509156_breast_breast_B17_B17_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509157_breast_breast_B16_B16_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509158_breast_breast_B15_B15_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509159_breast_breast_B14_B14_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509160_breast_breast_B2_B2_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509161_breast_breast_B13_B13_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509162_breast_breast_B12_B12_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509163_breast_breast_B11_B11_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509164_breast_breast_B10_B10_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509165_breast_breast_B4_B4_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad
  GSM9509166_breast_breast_B9_B9_rctd_annotated_cytotoxic

ValueError: Sampling selected no rows from /work/magroup/skrieger/Syndri/data/datasets/breast_xenium/results/responsiveness/predictions/GSM9509156_breast_breast_B17_B17_rctd_annotated_cytotoxic_responsiveness_predictions.h5ad

## Check sampled neighborhoods

This view shows up to 100 distinct representative contexts, so you can see which cell-type compositions were sampled.

In [ ]:
context_rows = adata.obs[['tissue', 'context_source_cell_id', 'sampling_repeat', 'is_representative_context']].copy()
context_rows['row_position'] = range(adata.n_obs)
context_rows = context_rows.loc[context_rows['is_representative_context'] & context_rows['sampling_repeat'].eq(context_rows['sampling_repeat'].min())]
context_rows = context_rows.drop_duplicates(['tissue', 'context_source_cell_id']).head(100)
context_matrix = adata.obsm['neighborhood_proportions'].iloc[context_rows['row_position'].to_numpy()].copy()
context_matrix.index = context_rows['context_source_cell_id'].to_numpy()
active_types = context_matrix.columns[context_matrix.max(axis=0) > 0]
if len(context_matrix) and len(active_types):
    fig, ax = plt.subplots(figsize=(max(7, len(active_types) * 0.7), 6))
    sns.heatmap(context_matrix[active_types].T, cmap='viridis', vmin=0, vmax=1, ax=ax, cbar_kws={'label': 'Neighbor proportion'})
    ax.set(xlabel='Representative contexts', ylabel='Neighbor cell type')
    ax.set_xticks([])
    plt.tight_layout()
    plt.show()

## Calculate responsiveness

`responses` has one row per query/context/gene with a change from that query's original context in the same repeat. `slopes` has one row per query/repeat/gene/neighbor type; its value describes the association with a 10 percentage-point change in neighbor proportion. Proportions are compositional, so these slopes are descriptive.

In [ ]:
responses, slopes = analyze_results(
    adata, genes=GENES, n_auto_genes=N_AUTO_GENES, logit=LOGIT,
)
print(f'{len(responses):,} response rows; {len(slopes):,} slope estimates')
display(responses.head())
display(slopes.head())

## Plots

The first plot shows gene-level changes from the original context. The clustered map summarizes median slopes across query cells and repeats, with a gene annotation bar when `Annotation_DB` is available.

In [ ]:
fig, ax = plot_response_distribution(responses, logit=LOGIT)
ax.set_title(f'{CELL_TYPE}: change from original neighborhood')
plt.show()

grid = plot_slope_clustermap(slopes, genes=GENES, annotation_col='Annotation_DB', logit=LOGIT)
grid.fig.suptitle(f'{CELL_TYPE}: within-query context associations', y=1.02)
grid.fig.set_dpi(200)
plt.show()

In [ ]:
# Inspect one gene against one varying neighbor type. Change these labels to explore others.
if not slopes.empty:
    plot_gene = slopes['gene'].iloc[0]
    plot_type = slopes['neighbor_cell_type'].value_counts().index[0]
    representative = adata.obs['is_representative_context'].astype(bool).to_numpy()
    gene_rows = responses.loc[responses['gene'].eq(plot_gene) & responses['is_representative_context']].copy()
    composition = adata.obsm['neighborhood_proportions'].loc[:, plot_type].to_numpy()[representative]
    gene_rows['neighbor_proportion'] = composition
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.scatterplot(data=gene_rows.sample(min(5000, len(gene_rows)), random_state=RANDOM_SEED),
                    x='neighbor_proportion', y='value', hue='tissue', alpha=0.25, s=18, ax=ax)
    ax.set(xlabel=f'Proportion {plot_type}', ylabel=f'{plot_gene} {"logit" if LOGIT else "probability"}')
    plt.tight_layout()
    plt.show()